# Configured solver diagnostics example

This notebook is a public configured-workflow example for package-generated configured solver diagnostics artifacts. It inspects `solver_diagnostics.json`, `solver_diagnostics.csv`, and the report/index links that expose those existing artifacts.

Guardrails: this is not an empirical validation, calibration, solver-quality threshold, empirical comparison, thermodynamic enforcement, or biology example. It adds no solver behavior change, no numerical quality thresholds, no validation/calibration evidence, no inferred scientific values, no hidden notebook science, and no biology claim. The diagnostics are recorded metadata for inspectability only.


In [ ]:
import csv
import json
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from fungal_model import load_model_config, run_configured_model
from fungal_model.api.report import write_virtual_experiment_report
from fungal_model.workflows import ConfiguredInputLoader, ConfiguredOutputWriter, ConfiguredProcessAssembler

SOURCE_CONFIG = ROOT / "data" / "model_configs" / "toy_homogeneous_ab.yml"
OUTPUT_ROOT = Path(os.environ.get("FUNGMOD_NOTEBOOK_OUTPUT_ROOT", str(ROOT / "notebooks" / "examples" / "Outputs")))
OUTPUT = OUTPUT_ROOT / "19_solver_diagnostics_example"
AVAILABLE_OUTPUT = OUTPUT / "available_metadata"
NO_METADATA_OUTPUT = OUTPUT / "header_only_no_metadata"

assert SOURCE_CONFIG.exists()


## Normal configured-output solver metadata path

`run_configured_model(...)` owns model loading, assembly, execution, validation, and configured output writing. The notebook reads the emitted solver diagnostics artifacts after the package writes them. The row values are existing run metadata, solver settings, time-grid counts, state counts, and process counts; this notebook does not score solver quality or define pass/fail thresholds.


In [ ]:
available_result = run_configured_model(SOURCE_CONFIG, output_dir=AVAILABLE_OUTPUT)

available_diagnostics = json.loads((AVAILABLE_OUTPUT / "solver_diagnostics.json").read_text(encoding="utf-8"))
with (AVAILABLE_OUTPUT / "solver_diagnostics.csv").open(newline="", encoding="utf-8") as handle:
    available_rows = list(csv.DictReader(handle))
available_manifest = json.loads((AVAILABLE_OUTPUT / "output_manifest.json").read_text(encoding="utf-8"))

available_summary = {
    "json_status": available_diagnostics["status"],
    "metadata_available": available_diagnostics["metadata_available"],
    "csv_row_count": len(available_rows),
    "solver_backend": available_rows[0]["solver_backend"],
    "solver_method": available_rows[0]["solver_method"],
    "configured_time_evaluation_count": available_rows[0]["configured_time_evaluation_count"],
    "result_time_point_count": available_rows[0]["result_time_point_count"],
    "allowed_use": available_rows[0]["allowed_use"],
}

assert available_result.solver_metadata["success"] is True
assert available_diagnostics["kind"] == "configured_solver_diagnostics"
assert available_diagnostics["status"] == "available"
assert available_diagnostics["metadata_available"] is True
assert available_diagnostics["row_count"] == len(available_rows) == 1
assert available_result.solver_metadata["backend"]
assert available_rows[0]["solver_backend"] == available_result.solver_metadata["backend"]
assert available_rows[0]["metadata_available"] == "True"
assert "not validation, calibration" in available_rows[0]["allowed_use"]
assert "does not infer scientific values" in available_rows[0]["interpretation_guardrail"]
assert "solver_diagnostics.json" in available_manifest["files"]
assert "solver_diagnostics.csv" in available_manifest["files"]
available_summary


## Header-only no-metadata guardrail

The configured output writer also has an explicit header-only guardrail when solver metadata is absent: JSON reports `status: unavailable`, and `solver_diagnostics.csv` keeps the expected header without row-level diagnostics. This block uses package workflow helpers to write that existing guardrail path after clearing metadata on a package-generated result. It is a software-contract demonstration, not a normal scientific run and not a reason to treat missing metadata as zero or as a quality score.


In [ ]:
config = load_model_config(SOURCE_CONFIG)
inputs = ConfiguredInputLoader().load(config)
assembly = ConfiguredProcessAssembler().assemble(config, inputs)
no_metadata_result = assembly.model.run(
    initial_state=inputs.initial_state,
    t_span=inputs.t_span,
    t_eval=inputs.t_eval,
    label=config.mode,
    name=config.name,
)
no_metadata_result.solver_metadata = {}

ConfiguredOutputWriter().write_result_bundle(
    config=config,
    inputs=inputs,
    decisions=assembly.decisions,
    result=no_metadata_result,
    output_dir=NO_METADATA_OUTPUT,
)

no_metadata_diagnostics = json.loads((NO_METADATA_OUTPUT / "solver_diagnostics.json").read_text(encoding="utf-8"))
with (NO_METADATA_OUTPUT / "solver_diagnostics.csv").open(newline="", encoding="utf-8") as handle:
    no_metadata_reader = csv.DictReader(handle)
    no_metadata_rows = list(no_metadata_reader)
no_metadata_manifest = json.loads((NO_METADATA_OUTPUT / "output_manifest.json").read_text(encoding="utf-8"))
no_metadata_fieldnames = list(no_metadata_reader.fieldnames or [])

no_metadata_summary = {
    "json_status": no_metadata_diagnostics["status"],
    "metadata_available": no_metadata_diagnostics["metadata_available"],
    "csv_row_count": len(no_metadata_rows),
    "csv_has_solver_backend_column": "solver_backend" in no_metadata_fieldnames,
    "csv_has_nfev_column": "nfev" in no_metadata_fieldnames,
    "missing_metadata_fields": no_metadata_diagnostics["missing_metadata_fields"],
}

assert no_metadata_diagnostics["kind"] == "configured_solver_diagnostics"
assert no_metadata_diagnostics["status"] == "unavailable"
assert no_metadata_diagnostics["metadata_available"] is False
assert no_metadata_diagnostics["row_count"] == 0
assert no_metadata_rows == []
assert "solver_backend" in no_metadata_fieldnames
assert "nfev" in no_metadata_fieldnames
assert "solver_diagnostics.json" in no_metadata_manifest["files"]
assert "solver_diagnostics.csv" in no_metadata_manifest["files"]
no_metadata_summary


## Report and index visibility

The configured-output report utilities can link the existing solver diagnostics artifacts. The report section is presentation-only: it repeats recorded status and row metadata so a reader can find the JSON/CSV files, without changing solver behavior or interpreting solver metadata as validation/calibration evidence.


In [ ]:
report_path = write_virtual_experiment_report(
    table_dir=AVAILABLE_OUTPUT,
    output_dir=AVAILABLE_OUTPUT / "report",
    include_html=True,
    include_index=True,
)
report_text = report_path.read_text(encoding="utf-8")
html_text = report_path.with_suffix(".html").read_text(encoding="utf-8")
index_text = report_path.with_name("index.html").read_text(encoding="utf-8")

report_summary = {
    "report_path": str(report_path),
    "report_links_solver_json": "solver_diagnostics.json" in report_text,
    "html_links_solver_csv": "solver_diagnostics.csv" in html_text,
    "index_links_solver_json": "solver_diagnostics.json" in index_text,
}

assert "## Configured solver diagnostics" in report_text
assert "existing configured-output `solver_diagnostics.json` and `solver_diagnostics.csv` artifacts only" in report_text
assert "do not change solver behavior" in report_text
assert "define numerical quality thresholds" in report_text
assert "validation/calibration evidence" in report_text
assert "solver_diagnostics.json" in html_text
assert "solver_diagnostics.csv" in html_text
assert "solver_diagnostics.json" in index_text
assert "solver_diagnostics.csv" in index_text
report_summary


## What this proves and what it does not prove

This example proves that a configured output bundle records solver diagnostics artifacts and that the report/index paths can expose those existing files for inspection. It also shows the explicit header-only behavior when solver metadata is unavailable.

It does not add solver behavior, numerical quality thresholds, validation data, calibration, empirical comparison, thermodynamic enforcement, inferred scientific values, biology, hidden notebook science, or silent fallback constants.
